# Librerias y configuración

In [1]:
import os, datetime as dt, requests, json
import pandas as pd
#from datetime import datetime as dt
from google.colab import userdata

In [2]:
# Acceder al los tokens privados de acceso

tok = userdata.get("BEARER_TOKEN")  # Acepta el permiso cuando te lo pida
assert tok, "Falta BEARER_TOKEN. Define la variable o pégala temporalmente."
HEADERS = {"Authorization": f"Bearer {tok}"}


In [3]:
# Algunas cuentas responden en api.x.com; otras en api.twitter.com (legado)
BASE_URLS = ["https://api.x.com/2", "https://api.twitter.com/2"]

# Ping simple (usuario público)

Intentaremos obtener datos de un usuario público. Si devuelve 200, el token se acepta y los headers están bien.

In [4]:
def pick_base_url():
    for base in BASE_URLS:
        try:
            # Probar existencia del endpoint con una llamada simple
            url = f"{base}/users/by/username/Claudiashein"
            r = requests.get(url, headers=HEADERS, timeout=20)
            if r.status_code in (200, 401, 403, 404):  # el endpoint existe
                return base
        except Exception:
            pass
    raise RuntimeError("No se detectó un BASE_URL válido (api.x.com o api.twitter.com).")

base = pick_base_url()
print("Usando base:", base)

Usando base: https://api.x.com/2


In [5]:
url_user = f"{base}/users/by/username/Claudiashein"  # usuario público estable
r = requests.get(url_user, headers=HEADERS, timeout=20)
print("HTTP Status:", r.status_code)

HTTP Status: 200


In [6]:
if r.status_code == 200:
    data = r.json().get("data", {})
    print("OK ✅ Token aceptado. Usuario obtenido:")
    print(json.dumps({k: data.get(k) for k in ["id","name","username","created_at"]}, indent=2))
elif r.status_code == 401:
    print("❌ 401 Unauthorized: revisa que el BEARER_TOKEN sea correcto y vigente.")
elif r.status_code == 403:
    print("⚠️ 403 Forbidden: token válido pero sin permiso para este recurso en tu plan.")
else:
    print("Respuesta:", r.text[:500])

OK ✅ Token aceptado. Usuario obtenido:
{
  "id": "591361197",
  "name": "Claudia Sheinbaum Pardo",
  "username": "Claudiashein",
  "created_at": null
}


# Búsqueda mínima (10 resultados recientes)

Verificamos que también podamos hacer una consulta básica al endpoint de búsqueda.
Si el plan/permiso contratado en X, no lo habilita, verás 403; si funciona, verás 200 y un conteo.

In [7]:
# Ventana de tiempo corta (últimas ~24h) y consulta ligera en español
end_time = dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
start_time = (dt.datetime.utcnow().replace(microsecond=0) - dt.timedelta(days=1)).isoformat() + "Z"

print(f"{start_time} - {end_time}")

2025-10-07T23:57:21Z - 2025-10-08T23:57:21Z


/tmp/ipython-input-1540126615.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time = dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
/tmp/ipython-input-1540126615.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time = (dt.datetime.utcnow().replace(microsecond=0) - dt.timedelta(days=1)).isoformat() + "Z"


In [8]:
params = {
    "query": '(seguridad OR "seguridad pública") lang:es -is:retweet',
    "max_results": 10,
    "start_time": start_time,
    "end_time": end_time,
    "tweet.fields": "id,text,author_id,created_at,lang"
}

url_search = f"{base}/tweets/search/recent"
r = requests.get(url_search, headers=HEADERS, params=params, timeout=30)

print("HTTP Status:", r.status_code)

HTTP Status: 400


In [9]:
if r.status_code == 200:
    payload = r.json()
    n = len(payload.get("data", []))
    print(f"OK: Búsqueda reciente funcionando. Tweets obtenidos: {n}")
    if n:
        sample = payload["data"][0]
        print("Ejemplo (primer tweet):")
        print(json.dumps({k: sample.get(k) for k in ["id","text","created_at","lang"]}, indent=2, ensure_ascii=False))
elif r.status_code == 401:
    print("401 Unauthorized: token inválido/expirado o header mal formado.")
elif r.status_code == 403:
    print("403 Forbidden: tu plan o permisos no permiten /tweets/search/recent.")
    print("   Sugerencia: valida en tu dashboard de X si el endpoint está habilitado en Basic.")
else:
    print("Respuesta:", r.text[:500])

Respuesta: {"errors":[{"parameters":{"end_time":["2025-10-08T23:57Z"]},"message":"Invalid 'end_time':'2025-10-08T23:57Z'. 'end_time' must be a minimum of 10 seconds prior to the request time."}],"title":"Invalid Request","detail":"One or more parameters to your request was invalid.","type":"https://api.twitter.com/2/problems/invalid-request"}


# Recolección de Datos

* params: son los parámetros de la URL (query string) que le dices al endpoint: qué buscar, cuántos resultados, qué campos, etc.

* headers: incluyen la autenticación: {"Authorization": "Bearer TU_TOKEN"}.

* r = requests.get(...): hace la petición HTTP.

* payload = r.json():

  * “Payload” es el nombre de una variable donde guardamos el cuerpo (body) de la respuesta HTTP en formato JSON.

  * En HTTP se suele llamar payload al contenido que viaja en la petición o respuesta. Es decir, aquí almacenamos la respuesta a la pregunta/consulta hecha a X.
  * Estructura del payload de X:

     * data: lista de tweets (cada uno con id, text, etc.).

     * includes: “tablas” extra relacionadas; en este caso, users si pedimos expansions="author_id".

     * meta: metadatos de la búsqueda (result_count, next_token para paginación, etc.).

* next_token: si existe, significa que hay otra página de resultados; por eso hacemos un while hasta agotar páginas o llegar al total_limit.

* users_by_id: indexamos usuarios por id para poder imprimir @username junto al tweet.

In [10]:
# --- Consulta de ejemplo (ajústala si quieres) ---
QUERY = '(seguridad OR "seguridad pública") lang:es -is:retweet'

consideramos una búsqueda en que:

* ( ) agrupa términos.

* OR busca cualquiera de los términos.

* "seguridad pública" obliga a la frase exacta.

* lang:es filtra a español.

* -is:retweet excluye retuits (menos ruido).

Podemos ajustar nuestra query conforme necesitemos:

1.  Cobertura semántica

- Añade sinónimos y variaciones:
("robo" OR "robos" OR "asalto" OR "asaltos" OR "extorsión" OR "balacera")

- Incluye hashtags y formas coloquiales:
(#seguridad OR "me asaltaron" OR "me robaron")

2. Ruido / calidad

- Excluir respuestas y citas si quieres solo publicaciones “originales”:
-is:reply -is:quote

- Filtrar por contenido:

  - Solo enlaces (notas/medios): has:links

  - Solo testimonios (sin enlaces): -has:links

  - Con multimedia: has:images OR has:videos

- Quitar términos que contaminen (deporte, política no relacionada, sorteos, etc.):
-fútbol -árbitro -penal -giveaway

3. Fuente

- Cuentas concretas (autoridades/medios):
(from:SSC_CDMX OR from:FiscaliaCDMX OR from:LaRazon_mx)

- Conversaciones hacia/desde una cuenta:
to:SSC_CDMX / from:SSC_CDMX

4.  Ubicación (si el tipo de cuenta del token lo permite)

- País: place_country:MX

- Lugares específicos requieren expansions=geo.place_id y place.fields=....
Si no tienes geo habilitado, aproxima con texto:
(CDMX OR "Ciudad de México" OR "Edomex" OR "GAM" OR "Iztapalapa").


5. Intención analítica

- Incidentes vs percepción:

  - Incidentes: (robo OR asalto OR "robo a casa")

  - Percepción: ("me asaltaron" OR "me robaron" OR "tengo miedo" OR inseguridad)

- Medir difusión: añade métricas mínimas solo en análisis posterior (en la API la criba por likes/RT no siempre está disponible; mejor filtrar luego por public_metrics).

6. Tiempo

- En v2 de la API de X no se debe usar el parámetro until:/since: en la query; controla ventana con start_time y end_time.

In [11]:
def recent_all(base, headers, query, start_time, end_time, total_limit=300):
    """
    Recolecta varias páginas del endpoint /2/tweets/search/recent hasta total_limit.
    Devuelve:
      - all_tweets: lista con todos los tweets (data)
      - users_by_id: diccionario {user_id: objeto usuario} si pedimos expansions
      - all_pages: lista con el JSON crudo (payload) de cada página
    """
    url = f"{base}/tweets/search/recent"
    params = {
        "query": query,
        "max_results": 100,  # máximo por página
        "start_time": start_time,
        "end_time": end_time,
        "tweet.fields": "id,text,author_id,created_at,lang,public_metrics",
        "expansions": "author_id",           # para obtener info de usuario
        "user.fields": "id,username,name"    # qué campos de usuario queremos
    }

    next_token = None
    all_tweets, all_pages = [], []
    users_by_id = {}

    while len(all_tweets) < total_limit:
        if next_token:
            params["next_token"] = next_token
        else:
            params.pop("next_token", None)

        r = requests.get(url, headers=headers, params=params, timeout=30)
        if r.status_code != 200:
            print("HTTP Status:", r.status_code, r.text[:400])
            break

        payload = r.json() # aqui el "PAYLOAD": cuerpo JSON de la respuesta HTTP
        all_pages.append(payload)

        # Estructura típica: data / includes / meta
        tweets = payload.get("data", [])
        includes = payload.get("includes", {})
        users = includes.get("users", [])

        # Indexamos usuarios por id (para imprimir @username)
        for u in users:
            users_by_id[u["id"]] = u

        all_tweets.extend(tweets)

        # Paginación
        next_token = payload.get("meta", {}).get("next_token")
        if not next_token or not tweets:
            break

    return all_tweets, users_by_id, all_pages


In [12]:
# Ejecutar recolección
 # Ajustar total_limit conforme se requiera
all_tweets, users_by_id, pages = recent_all(base, HEADERS, QUERY,
                                            start_time, end_time,
                                            total_limit=300)


HTTP Status: 400 {"errors":[{"parameters":{"end_time":["2025-10-08T23:57Z"]},"message":"Invalid 'end_time':'2025-10-08T23:57Z'. 'end_time' must be a minimum of 10 seconds prior to the request time."}],"title":"Invalid Request","detail":"One or more parameters to your request was invalid.","type":"https://api.twitter.com/2/problems/invalid-request"}


In [13]:
# Imprimir el payload JSON completo de la PRIMERA página ---
if pages:
    print("\n=== Payload (JSON) de la primera página ===")
    print(json.dumps(pages[0], ensure_ascii=False, indent=2))

In [14]:
# Impresión "bonita" de TODO lo recolectado
print(f"Total de tweets recolectados: {len(all_tweets)}\n")
for i, t in enumerate(all_tweets, 1):
    user = users_by_id.get(t["author_id"], {})
    handle = "@" + user.get("username", t["author_id"])
    created = t.get("created_at", "")
    tid = t.get("id", "")
    text = (t.get("text", "") or "").replace("\n", " ")
    print(f"{i:03d}. [{created}] {handle} (id:{tid})")
    print("     ", text)


Total de tweets recolectados: 0



# Crear dataframe de los datos colectados

In [15]:
# Los datos colectados están en formato JSON, debemos ir extrayendo
# cada campo e ir "montando" el dataframe

# 1) Normalizar listas de dicts a DataFrames
df_t = pd.json_normalize(all_tweets, sep=".")
df_u = pd.json_normalize(list(users_by_id.values()), sep=".") if users_by_id else pd.DataFrame()

# 2) Renombrar columnas de usuarios para no colisionar
if not df_u.empty:
    df_u = df_u.rename(columns={
        "id":"user.id",
        "name":"user.name",
        "username":"user.username",
        # agrega más campos en caso que hayan sido solicitados en user.fields
    })

# 3) Unir tweets con usuarios por author_id ↔ user.id
if not df_u.empty and "author_id" in df_t.columns:
    df = df_t.merge(
        df_u[["user.id","user.name","user.username"]],
        left_on="author_id", right_on="user.id", how="left"
    )
else:
    df = df_t.copy()

# 4) Tipos útiles y orden de columnas
if "created_at" in df.columns:
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

# Orden sugerido (si existen)
front = [c for c in [
    "id","text","created_at","lang","author_id",
    "user.username","user.name",
    "public_metrics.like_count","public_metrics.reply_count",
    "public_metrics.retweet_count","public_metrics.quote_count"
] if c in df.columns]
df = df[front + [c for c in df.columns if c not in front]]

# 5) Deduplicar por id, por si hubo solapamiento entre páginas
if "id" in df.columns:
    df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

print("Filas:", len(df), "| Columnas:", len(df.columns))
df.head(10)


Filas: 0 | Columnas: 0


""


In [16]:
# ---- Switch de entorno: "colab" o "local"
RUN_ENV = "colab"   # <-- cambia a "local" si no usas Google Colab

# ---- Rutas base
if RUN_ENV.lower() == "colab":
    try:
        from google.colab import drive
        drive.mount("/drive")
    except Exception as e:
        print("Aviso: no se pudo montar Drive automáticamente:", e)
    # Ajusta esta ruta a tu carpeta del proyecto en Drive
    BASE_DIR = "/drive/My Drive/Colab Notebooks/reto-Thales/"
else:
    # Ajusta a tu ruta local del proyecto
    BASE_DIR = "/path-de-computador-local/"

os.makedirs(BASE_DIR, exist_ok=True)

Mounted at /drive


In [17]:
if df.empty:
    print("No hay datos para exportar. Revisa la consulta o la ventana temporal.")
else:
    stamp = dt.utcnow().strftime("%Y%m%dT%H%M%SZ")
    out_csv = f"x_seguridad_cdmx_colecta_{stamp}.csv"
    df.to_csv(BASE_DIR + "data/" + out_csv, index=False, encoding="utf-8")
    print(f"CSV guardado: {out_csv}")
    print("Columnas:", list(df.columns))


No hay datos para exportar. Revisa la consulta o la ventana temporal.
